# Practical Session 01
## Part I - Instructor-led mini-lab
### Example 1 - Why not just use Python lists?
Take a look at the code below before running it.

Questions:

1. Will the two expressions produce the same result?
2. Which object represents a mathematical vector more naturally?
3. What does multiplication mean in each case?

In [ ]:
import sys
import time
import numpy as np

In [ ]:
python_values = [1, 2, 3]
numpy_values = np.array([1, 2, 3])

print(2 * python_values)
print(2 * numpy_values)

Next, compare approximate storage requirements:

In [ ]:
n = 30_000_000

values_list = list(range(n))
values_array = np.arange(n, dtype=np.int64)

estimated_list_size = (
    sys.getsizeof(values_list)
    + n * sys.getsizeof(values_list[0])
)

print(f"Estimated list size: {estimated_list_size / 1024**2:.1f} MB")
print(f"NumPy array size:    {values_array.nbytes / 1024**2:.1f} MB")

The list estimate is approximate but useful. It shows the main issue: a list of Python integers is not stored as raw integer data. It is a container of references to Python objects. For large scientific arrays, this difference quickly becomes important.

Now compare a simple numerical transformation:

$$
y_i = 3x_i + 1.
$$

For the Python list, we need to loop over elements explicitly. For the NumPy array, we apply the expression to the whole array

In [ ]:
n = 3_000_000

values_list = list(range(n))
values_array = np.arange(n, dtype=np.int64)

def transform_list(values):
    return [3 * x + 1 for x in values]

def transform_array(values):
    return 3 * values + 1

def best_average_time(function, repeats=3, number=5):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        for _ in range(number):
            function()
        stop = time.perf_counter()
        times.append((stop - start) / number)
    return min(times)

list_time = best_average_time(lambda: transform_list(values_list))
array_time = best_average_time(lambda: transform_array(values_array))

print(f"List comprehension: {1e3 * list_time:8.3f} ms")
print(f"NumPy expression:   {1e3 * array_time:8.3f} ms")
print(f"Speed-up factor:    {list_time / array_time:8.1f} x")

## Example 2 - Reading an array through its axes
A small collection of detector hits is stored as:

```text
hits[event, hit, component]
```

Before running each expression below, predict the shape of the result.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

# axes: event, hit, spatial component
hits = rng.normal(size=(4, 6, 3))

print("hits.shape:", hits.shape)
print("hits[0].shape:          ", hits[0].shape)
print("hits[:, 0].shape:       ", hits[:, 0].shape)
print("hits[..., 2].shape:     ", hits[..., 2].shape)
print("hits.mean(axis=1).shape:", hits.mean(axis=1).shape)
print("hits.mean(axis=(0, 1)).shape:", hits.mean(axis=(0, 1)).shape)

Questions:

1. Which axis represents independent events?
2. Which axis represents hits inside one event?
3. Which axis represents the spatial component?
4. Which axis should disappear if we compute one centre per event?

In [ ]:
# One centre per event.
event_centres = hits.mean(axis=1)

print("event_centres.shape:", event_centres.shape)

assert event_centres.shape == (4, 3)

## Example 3 - Broadcasting a calibration model

Suppose that every hit slot has its own gain and every coordinate has its own offset.

We want to compute:

$$
h^{\mathrm{corrected}}_{eic}=\left(h_{eic}-o_c\right)g_i.
$$

Here:  
$e$ - event  
$i$ - hit  
$c$ - component

In [ ]:
sensor_gain = np.linspace(0.9, 1.1, 6)               # one value per hit
coordinate_offset = np.array([0.2, -0.1, 0.05])      # x, y, z

print("hits.shape:             ", hits.shape)
print("sensor_gain.shape:      ", sensor_gain.shape)
print("coordinate_offset.shape:", coordinate_offset.shape)

We need broadcast-ready shapes:

```text
hits               : (event, hit, component)
coordinate_offset  : (    1,   1, component)
sensor_gain        : (    1, hit,         1)
```

In [ ]:
corrected_hits = (
    (hits - coordinate_offset[None, None, :])
    * sensor_gain[None, :, None]
)

print("corrected_hits.shape:", corrected_hits.shape)

assert corrected_hits.shape == hits.shape

Now add one event-dependent shift only to the $z$ component:

$$
h^{\mathrm{shifted}}_{ei2}=h^{\mathrm{corrected}}_{ei2}-d_e.
$$

In [ ]:
event_shift = np.array([0.0, 0.2, -0.1, 0.05])

shifted_hits = corrected_hits.copy()
shifted_hits[..., 2] -= event_shift[:, None]

print("event_shift[:, None].shape:", event_shift[:, None].shape)
print("shifted_hits.shape:        ", shifted_hits.shape)

assert shifted_hits.shape == corrected_hits.shape

## Example 4 - A batched quadratic form

For each particle, let:

$$
p = (E, p_x, p_y, p_z).
$$

With the Minkowski metric

$$
\eta = \mathrm{diag}(1, -1, -1, -1),
$$

the invariant mass satisfies:

$$
m_b^2 = p_{bi}\eta_{ij}p_{bj}.
$$

The index $b$ labels particles and should remain. The indices $i,j$ are contracted.

In [ ]:
spatial_momentum = rng.normal(size=(5, 3))
particle_mass = rng.uniform(0.2, 2.0, size=5)

energy = np.sqrt(
    particle_mass**2
    + np.sum(spatial_momentum**2, axis=1)
)

four_momentum = np.column_stack((energy, spatial_momentum))
metric = np.diag([1.0, -1.0, -1.0, -1.0])

print("four_momentum.shape:", four_momentum.shape)
print("metric.shape:       ", metric.shape)

In [ ]:
mass_squared = np.einsum(
    "bi,ij,bj->b",
    four_momentum,
    metric,
    four_momentum,
)

reconstructed_mass = np.sqrt(mass_squared)

print("original masses:      ", particle_mass)
print("reconstructed masses: ", reconstructed_mass)

assert mass_squared.shape == (5,)
assert np.allclose(reconstructed_mass, particle_mass)

The same result can also be obtained in two steps: first apply the metric, then reduce over the component axis.

In [ ]:
metric_times_p = four_momentum @ metric
mass_squared_alt = np.sum(metric_times_p * four_momentum, axis=1)

print(mass_squared_alt)

assert np.allclose(mass_squared_alt, mass_squared)

Questions:

1. Which index labels independent particles?
2. Which indices disappear?
3. Why is the result one-dimensional?

## Example 5 - The view that changes the original data

Some NumPy operations create views of the same data. This is efficient, but it can also lead to accidental modification of the original array.

In [ ]:
raw_data = np.arange(24.0).reshape(4, 6)
window = raw_data[:, 2:5]

print("raw_data before:")
print(raw_data)

print("\nwindow:")
print(window)

print("\nShare memory?", np.shares_memory(raw_data, window))

In [ ]:
window[0, 0] = -999.0

print("raw_data after modifying window:")
print(raw_data)

The slice `raw_data[:, 2:5]` is a view. It does not own independent data.

To work safely on an independent array, use `.copy()`.

In [ ]:
raw_data = np.arange(24.0).reshape(4, 6)
safe_window = raw_data[:, 2:5].copy()

safe_window[0, 0] = -999.0

print("safe_window:")
print(safe_window)

print("\nraw_data:")
print(raw_data)

print("\nShare memory?", np.shares_memory(raw_data, safe_window))

## Part II - Independent work

### Task 1 - A Calibration Pipeline with Three Independent Axes

A detector records a short signal from several channels during each event. The array

```text
raw[event, channel, sample]
```
has shape `(12, 7, 64)`.

Each channel has its own pedestal $p_c$ and gain $g_c$. In addition, every event has a common drift $d_e$. The calibrated signal is

$$
s_{ecs} = (r_{ecs}-p_c-d_e)g_c
$$
Use the following data:

In [ ]:
import numpy as np

rng = np.random.default_rng(2026)

n_events = 12
n_channels = 7
n_samples = 64

raw = rng.normal(
    loc=1000.0,
    scale=20.0,
    size=(n_events, n_channels, n_samples),
)

pedestal = rng.normal(
    loc=995.0,
    scale=2.0,
    size=n_channels,
)

gain = rng.uniform(
    0.8,
    1.2,
    size=n_channels,
)

event_drift = rng.normal(
    0.0,
    1.0,
    size=n_events,
)

Your tasks are:

1. Write one vectorized expression that calculates the calibrated signal. Do not use loops or explicitly repeat any calibration array.
2. Add comments showing the broadcast-ready shape of `pedestal`, `gain`, and `event_drift`.
3. Verify that the calibrated signal has the same shape as `raw`.
4. Calculate the temporal mean of every event-channel trace. The result should have shape (`n_events`, `n_channels`).
5. Subtract this temporal mean from each corresponding trace while preserving the original three-dimensional shape.
6. Verify numerically that the temporal means of the centred traces are approximately zero.
7. Find the event index, channel index, and sample index at which the centred signal has the largest absolute value.

**Optional extension**: Package the calibration operation in a function that checks all input shapes and raises an informative error when they are inconsistent.

In [ ]:
#####################
# Your solution here
#####################

## Task 2 - The Case of the Corrupted Raw Data

A preprocessing step extracts samples 3–8 from every detector trace and centres them:

In [ ]:
raw = np.arange(6 * 12, dtype=float).reshape(6, 12)
raw_before_processing = raw.copy()

window = raw[:, 3:9]
window -= window.mean(axis=1, keepdims=True)

After this operation, a colleague notices that raw no longer contains the original measurements.

Your tasks are:

1. Compare `raw` with `raw_before_processing` and identify which entries changed.
2. Explain why modifying `window` also modified `raw`.
3. Repair the preprocessing step so that the extracted window can be modified safely while `raw` remains unchanged.
4. Use `np.shares_memory` to investigate whether each of the following is normally a view or a copy:

In [ ]:
raw[:, 2:8]
raw[:, ::2]
raw.T
raw.reshape(-1)
raw.T.reshape(-1)
raw[:, [2, 4, 6]]
raw[raw > 20]
raw + 0.0

5. For every expression, record its shape and whether it shares memory with `raw`.
6. Create an independent centred window, flatten it into a one-dimensional solver state, and then reconstruct its original two-dimensional shape.
7. Verify that reconstruction preserves all values exactly.

Conclude with two or three sentences explaining why an operation that is efficient because it avoids copying can also create a correctness problem.

In [ ]:
#####################
# Your solution here
#####################

## Task 3 - A Resonance Hidden in a Combinatorial Background

A simplified detector event contains two collections of approximately massless particle candidates with opposite electric charges. Each candidate is represented by a four-momentum

$$
p = (E, p_x, p_y, p_z)
$$

One positive-negative pair was generated from a particle with mass

$$
M_{\text{target}} = 91.1876
$$

The remaining combinations form a combinatorial background.

Use the following data-generation cell:

In [ ]:
import numpy as np

rng = np.random.default_rng(31415)
target_mass = 91.1876

def random_massless_four_momenta(n):
    directions = rng.normal(size=(n, 3))
    directions /= np.linalg.norm(
        directions,
        axis=1,
        keepdims=True,
    )

    energies = rng.uniform(10.0, 80.0, size=n)

    return np.column_stack(
        (energies, energies[:, None] * directions)
    )

positive = random_massless_four_momenta(28)
negative = random_massless_four_momenta(31)

direction = np.array([0.3, -0.4, 0.866025403784])
direction /= np.linalg.norm(direction)

energy = target_mass / 2.0

positive_signal = np.r_[energy, energy * direction]
negative_signal = np.r_[energy, -energy * direction]

positive = np.vstack((positive, positive_signal))
negative = np.vstack((negative, negative_signal))

positive = positive[rng.permutation(len(positive))]
negative = negative[rng.permutation(len(negative))]

metric = np.diag([1.0, -1.0, -1.0, -1.0])

For a positive candidate $i$ and a negative candidate $j$, define

$$
P_{ij} = p_i^{(+)} + p_j^{(-)}
$$

and

$$
m_{ij}^{2}=P_{ij}^{\mu}\,\eta_{\mu\nu}\,P_{ij}^{\nu},\qquad
\eta = \operatorname{diag}(1,-1,-1,-1).
$$

Your tasks are:

1. State the meaning of both axes of `positive` and `negative`.
2. Use broadcasting to construct the four-momentum of every possible positive-negative pair. The result should have shape
```text
(number of positive candidates,
 number of negative candidates,
 4)
```
3. Use `np.einsum` to calculate the complete matrix of invariant masses. Do not loop over candidate pairs.
4. Protect the square root from very small negative values caused by floating-point round-off.
5. Find the pair whose invariant mass is closest to `target_mass`.
6. Report:
    - the two candidate indices;
    - their reconstructed invariant mass;
    - the absolute mass difference from the target;
    - the magnitude of the total three-momentum of the pair.
7. Find all candidate pairs within 0.5 units of the target mass.
8. For every positive candidate, find the negative candidate that gives the closest mass to the target. Be careful to reduce over the correct axis.
9. Validate the invariant mass of the best pair using the direct scalar expression

$$
m^2 = E^2 - p_x^2 - p_y^2 - p_z^2
$$

No nested Python loops may be used.

**Optional extension**: Repeat the calculation after exchanging the positive and negative collections. Explain why the new mass matrix should be the transpose of the original one.

Useful NumPy tools may include `np.argmin`, `np.unravel_index`, `np.where`, and `np.clip`.

In [ ]:
#####################
# Your solution here
#####################